# Day 4: Train Multiple Linear Regression

**Theme:** Train a scikit-learn multiple linear regression model to predict `abs_g_mag` from selected Gaia-derived features.

Today we will:

- load the Day 3 feature-selection dataset
- choose a non-leaky multiple-feature set
- split rows into train and test sets with a fixed random seed
- fit a `StandardScaler` on the training features only
- train `LinearRegression` on the scaled training matrix
- inspect the learned intercept and coefficients
- compare actual and predicted `abs_g_mag` for example test rows
- save Day 4 predictions for Day 5 evaluation

The Week 2 model form is:

```text
prediction = w1*x1 + w2*x2 + ... + b
```

Here, each `w` is a learned weight for one selected Gaia feature, each `x` is the scaled feature value for one star, and `b` is the learned intercept.


## 1. Imports

We use scikit-learn for the train/test split, feature scaling, and linear regression model.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)


## 2. Load The Day 3 Feature Dataset

Day 4 normally starts from `gaia_day3_feature_selection.csv`. If that file is missing, the notebook rebuilds the required columns from the cleaned Gaia sample and Day 2 target function.


In [ ]:
project_root_candidates = [Path(".."), Path("."), Path("gaia-explorer")]
PROJECT_ROOT = next((path for path in project_root_candidates if (path / "src").exists()), None)

if PROJECT_ROOT is None:
    searched = "\n".join(str(path.resolve()) for path in project_root_candidates)
    raise FileNotFoundError(f"Could not find gaia-explorer/src. Searched:\n{searched}")

sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from src.data_clean import add_absolute_magnitude_target
from src.data_load import load_clean_gaia_sample

input_candidates = [
    Path("../data/processed/gaia_day3_feature_selection.csv"),
    Path("data/processed/gaia_day3_feature_selection.csv"),
    Path("gaia-explorer/data/processed/gaia_day3_feature_selection.csv"),
]

DATA_PATH = next((path for path in input_candidates if path.exists()), None)

if DATA_PATH is not None:
    feature_df = pd.read_csv(DATA_PATH)
    print(f"Loaded Day 3 feature data from: {DATA_PATH}")
else:
    clean_df, clean_path = load_clean_gaia_sample()
    target_df = add_absolute_magnitude_target(clean_df)
    feature_df = target_df[["source_id", "abs_g_mag", "bp_rp", "parallax_snr", "ra", "dec"]].copy()

    output_path = next((path for path in input_candidates if path.parent.exists()), input_candidates[0])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    feature_df.to_csv(output_path, index=False)

    DATA_PATH = output_path
    print(f"Rebuilt Day 3 feature data from {clean_path} and saved it to: {DATA_PATH}")

print("Feature dataset shape:", feature_df.shape)
feature_df.head()


## 3. Choose The Model Features

The target is:

```python
y = abs_g_mag
```

For the main Day 4 model, we use:

```python
model_features = ["bp_rp", "ra", "dec"]
```

Why not include every available column?

- `bp_rp` is the clean physical feature because color is tied to stellar temperature and brightness.
- `ra` and `dec` are diagnostic sky-position features. They make this a multiple linear regression example, but their coefficients should not be interpreted as physical luminosity causes.
- `parallax_snr` is useful for quality diagnostics, but it contains `parallax`. Since `abs_g_mag` is computed from `parallax`, we leave `parallax_snr` out of the main Day 4 model to avoid indirect leakage risk.
- `parallax`, `distance_pc`, and `phot_g_mean_mag` are excluded because they directly or indirectly define the target.


In [ ]:
target_column = "abs_g_mag"
model_features = ["bp_rp", "ra", "dec"]
diagnostic_only_features = ["parallax_snr"]
excluded_leakage_features = ["parallax", "distance_pc", "phot_g_mean_mag"]

required_columns = ["source_id", target_column] + model_features + diagnostic_only_features
missing_required_columns = [col for col in required_columns if col not in feature_df.columns]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

modeling_df = feature_df.dropna(subset=required_columns).copy()

feature_choice_table = pd.DataFrame(
    [
        {
            "role": "target",
            "columns": target_column,
            "use_in_day4_model": "Predict this",
            "reason": "Absolute Gaia G magnitude is the regression target.",
        },
        {
            "role": "main model features",
            "columns": ", ".join(model_features),
            "use_in_day4_model": "Yes",
            "reason": "Uses one physical color feature plus sky-position diagnostic features for multiple regression.",
        },
        {
            "role": "diagnostic only",
            "columns": ", ".join(diagnostic_only_features),
            "use_in_day4_model": "No",
            "reason": "Useful for quality checks, but parallax_snr carries indirect leakage risk.",
        },
        {
            "role": "excluded leakage features",
            "columns": ", ".join(excluded_leakage_features),
            "use_in_day4_model": "No",
            "reason": "These columns directly or indirectly define abs_g_mag.",
        },
    ]
)

print("Rows available for modeling:", len(modeling_df))
feature_choice_table


## 4. Create Train And Test Sets

The model should be trained on one part of the data and checked on held-out rows. We use a fixed random seed so the split is reproducible.

Important scaling rule: fit the scaler on the training set only, then use that same scaler to transform the test set. This avoids letting test-set information influence training.


In [ ]:
TEST_SIZE = 0.20
RANDOM_STATE = 42

X = modeling_df[model_features]
y = modeling_df[target_column]
metadata = modeling_df[["source_id", "parallax_snr"]].copy()

X_train, X_test, y_train, y_test, metadata_train, metadata_test = train_test_split(
    X,
    y,
    metadata,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

split_summary = pd.DataFrame(
    [
        {"split": "train", "rows": len(X_train), "feature_columns": X_train.shape[1]},
        {"split": "test", "rows": len(X_test), "feature_columns": X_test.shape[1]},
    ]
)

split_summary


## 5. Train `LinearRegression`

The model learns one coefficient per scaled feature plus an intercept.

Because the input features are standardized, each coefficient means:

> expected change in predicted `abs_g_mag` for a one-standard-deviation increase in that feature, holding the other features fixed.

Remember that magnitude is inverted: larger `abs_g_mag` means fainter.

In code, this step has three parts:

1. `model = LinearRegression()` creates the model object.
2. `model.fit(X_train_scaled, y_train)` learns the best coefficients and intercept from the training data.
3. `model.predict(...)` uses the learned formula to estimate `abs_g_mag` for training and test rows.

Training predictions tell us how well the model fits rows it already learned from. Test predictions are more important because they show how the model behaves on held-out stars it did not train on.


In [ ]:
# Create an empty linear regression model.
model = LinearRegression()

# Learn the coefficients and intercept from the scaled training features.
# y_train is not scaled, so predictions stay in real abs_g_mag units.
model.fit(X_train_scaled, y_train)

# Predict on the training rows to see how well the model fit the data it learned from.
y_train_pred = model.predict(X_train_scaled)

# Predict on held-out test rows to see how well the model generalizes to new stars.
y_test_pred = model.predict(X_test_scaled)

# The learned intercept is b; the learned coefficients are w1, w2, ...
intercept = model.intercept_
coefficients = model.coef_

print("Model trained")
print("Intercept:", intercept)
print("Number of coefficients:", len(coefficients))


## 6. Inspect The Intercept And Coefficients

The intercept is the prediction when all scaled features are `0`, which means each feature is at its training-set average.

The coefficient table connects each learned weight to one selected Gaia feature.

The learned formula is:

```text
predicted abs_g_mag =
    coefficient_for_bp_rp * scaled_bp_rp
  + coefficient_for_ra    * scaled_ra
  + coefficient_for_dec   * scaled_dec
  + intercept
```

Because the features are scaled, a coefficient tells us how much the prediction changes when that feature increases by one standard deviation while the other features stay fixed.

Example: if the `bp_rp` coefficient is positive, then redder stars are predicted to have larger `abs_g_mag`. Since larger magnitude means fainter, that means the model learned that redder/cooler stars tend to be fainter.

The intercept is the baseline prediction for an average star in the training set, because scaled feature value `0` means the feature is at its training-set average.


In [ ]:
coefficient_table = pd.DataFrame(
    {
        "feature": model_features,
        "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients),
        "interpretation_note": [
            "Physical color feature; positive means redder/cooler stars are predicted fainter.",
            "Diagnostic sky-position feature; do not treat as a physical luminosity cause.",
            "Diagnostic sky-position feature; do not treat as a physical luminosity cause.",
        ],
    }
).sort_values("absolute_coefficient", ascending=False)

intercept_table = pd.DataFrame(
    [{"parameter": "intercept", "value": intercept, "meaning": "Prediction when scaled features are at their training-set means."}]
)

print("Model intercept:")
display(intercept_table)

print("Learned coefficients:")
coefficient_table


## 7. Inspect Sample Predictions

These rows come from the held-out test set. Day 5 will evaluate the predictions more formally with MAE, RMSE, R-squared, and residual plots.


In [ ]:
predictions_df = metadata_test.copy()
predictions_df["actual_abs_g_mag"] = y_test
predictions_df["predicted_abs_g_mag"] = y_test_pred
predictions_df["residual"] = predictions_df["actual_abs_g_mag"] - predictions_df["predicted_abs_g_mag"]

for feature in model_features:
    predictions_df[feature] = X_test[feature]

sample_predictions = predictions_df[
    ["source_id", "bp_rp", "ra", "dec", "parallax_snr", "actual_abs_g_mag", "predicted_abs_g_mag", "residual"]
].head(10)

sample_predictions


## 8. Save Day 4 Model Outputs

We save the test-set predictions and coefficient table so Day 5 can focus on evaluation instead of retraining from scratch.


In [ ]:
prediction_output_candidates = [
    Path("../data/processed/gaia_day4_linear_regression_predictions.csv"),
    Path("data/processed/gaia_day4_linear_regression_predictions.csv"),
    Path("gaia-explorer/data/processed/gaia_day4_linear_regression_predictions.csv"),
]
coefficient_output_candidates = [
    Path("../data/processed/gaia_day4_linear_regression_coefficients.csv"),
    Path("data/processed/gaia_day4_linear_regression_coefficients.csv"),
    Path("gaia-explorer/data/processed/gaia_day4_linear_regression_coefficients.csv"),
]

PREDICTION_OUTPUT_PATH = next(
    (path for path in prediction_output_candidates if path.parent.exists()),
    prediction_output_candidates[0],
)
COEFFICIENT_OUTPUT_PATH = next(
    (path for path in coefficient_output_candidates if path.parent.exists()),
    coefficient_output_candidates[0],
)

PREDICTION_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
COEFFICIENT_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

predictions_df.to_csv(PREDICTION_OUTPUT_PATH, index=False)
coefficient_table.to_csv(COEFFICIENT_OUTPUT_PATH, index=False)

print(f"Saved Day 4 predictions to: {PREDICTION_OUTPUT_PATH}")
print("Prediction rows:", len(predictions_df))
print(f"Saved Day 4 coefficients to: {COEFFICIENT_OUTPUT_PATH}")
print("Feature names:", model_features)
print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)


## Reflection Questions And Starter Answers

1. Which feature has the largest learned coefficient?

   **Starter answer:** Check the coefficient table sorted by `absolute_coefficient`. The largest absolute value is the feature with the strongest linear weight in this scaled model.

2. Does the sign of the `bp_rp` coefficient make astronomical sense?

   **Starter answer:** A positive `bp_rp` coefficient means redder stars are predicted to have larger `abs_g_mag`. Since larger magnitude means fainter, that agrees with the main-sequence pattern where cooler/redder stars are often fainter.

3. What does a linear model assume about the relationship between color and absolute magnitude?

   **Starter answer:** It assumes a straight-line relationship after scaling: each one-unit change in a feature changes the prediction by a constant amount. The real color-magnitude diagram is curved and has multiple stellar populations, so this model is useful but physically incomplete.

4. Why are `ra` and `dec` coefficients hard to interpret physically?

   **Starter answer:** `ra` and `dec` describe sky position, not a star's internal physics. If they get nonzero coefficients, that may reflect sampling patterns or sky-region effects rather than a cause of luminosity.

5. Why did we leave `parallax_snr` out of the main Day 4 model?

   **Starter answer:** `parallax_snr` contains `parallax`, and `abs_g_mag` is computed from `parallax`. It is useful for diagnostics, but it carries indirect leakage risk as a normal predictor.
